# Chapter 7.1 n-step TD: MC와 TD(0) 사이의 다이얼

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter07_1_n_step_td.ipynb)

책 본문: [7.1 n-step TD](https://smhanlab.com/book-ml/kor/ml2/chapter07.html)

이 노트북은 7개 상태 random walk에서 **n-step TD**를 구현하고:

1. $n=1$이 정확히 TD(0)와 같은지를 검증하고,
2. **편향-분산 트레이드오프의 U자 곡선**(n에 따른 RMS 오차)을 재현하고,
3. U자 곡선의 최저점이 에피소드 예산에 따라 이동하는 것을 관찰합니다.

## 1. 환경: 7개 상태 random walk

상태 0과 6이 터미널이고, 상태 $1\sim5$는 50:50으로 좌우 이동(무작위 정책). 왼쪽 끝에 도달하면 보상 $-1$, 오른쪽 끝에 도달하면 $+1$, $\gamma=0.9$, 시작 상태 3.

참값(벨만방정식으로 정확히 구할 수 있음, 연습문제 2)은:

$$V^\pi(s) = [0,\ -0.5643,\ -0.2539,\ 0,\ +0.2539,\ +0.5643,\ 0]$$

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import random
import numpy as np

IMG = "/home/smhan/book-ml/kor/src/images"

def env_step(s, a):
    """random walk 환경. a: 0=left, 1=right (무작위 정책이라 a는 무시)."""
    ns = s + (1 if random.random() < 0.5 else -1)
    if ns == 0:
        return 0, -1.0, True   # 왼쪽 끝 도착: 보상 -1, 종료
    if ns == 6:
        return 6, 1.0, True    # 오른쪽 끝 도착: 보상 +1, 종료
    return ns, 0.0, False

V_TRUE = np.array([0.0, -0.5643, -0.2539, 0.0, 0.2539, 0.5643, 0.0])

## 2. n-step TD (online 관점)

$n$-step 리턴 $G_t^{(n)} = R_t + \gamma R_{t+1} + \cdots + \gamma^{n-1} R_{t+n-1} + \gamma^n V(s_{t+n})$ — 실제 보상 $n$개 + 나머지는 추정치.

online 관점의 핵심: 갱신 대상 `tau`가 현재 시점보다 $n-1$만큼 뒤처짐 (`tau = t - n + 1`) — $n$개 보상이 다 모여야 목표값을 계산할 수 있기 때문. 에피소드 종료 후(`t >= T`)에도 `tau == T-1`까지 "꼬리 갱신"(절단된 목표값)이 이어지므로, 에피소드당 갱신 횟수는 n과 무관하게 TD(0)와 항상 같다.

In [2]:
def n_step_td(n, n_episodes, alpha, gamma, n_states, start_state, seed=0):
    random.seed(seed)
    V = [0.0] * n_states
    for _ in range(n_episodes):
        states, rewards = [start_state], []
        s, T, t = start_state, float('inf'), 0
        while True:
            if t < T:
                a = random.randrange(2)  # 데모용 무작위 정책
                ns, r, done = env_step(s, a)
                states.append(ns)
                rewards.append(r)
                if done:
                    T = t + 1
                s = ns
            tau = t - n + 1  # tau번째 상태를 지금 갱신
            if tau >= 0:
                G = sum(gamma ** (i - tau) * rewards[i]
                        for i in range(tau, min(tau + n, len(rewards))))
                if tau + n < T:  # 윈도우 안에 에피소드가 안 끝났으면: 추정치 항 추가
                    G += gamma ** n * V[states[tau + n]]
                V[states[tau]] += alpha * (G - V[states[tau]])
            if tau == T - 1:
                break
            t += 1
    return np.array(V)

V_3 = n_step_td(n=3, n_episodes=10, alpha=0.1, gamma=0.9, n_states=7, start_state=3, seed=42)
print("n=3, 10에피소드 학습:", np.round(V_3, 3))
print("참값                 :", np.round(V_TRUE, 3))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(V_TRUE, "o-", color="tab:gray", label="True $V^\pi$")
ax.plot(V_3, "s--", color="tab:red", label="n-step TD estimate (n=3, 10 episodes)")
ax.axvline(0, color="k", lw=0.5, alpha=0.3); ax.axvline(6, color="k", lw=0.5, alpha=0.3)
ax.set(xlabel="State", ylabel="Value",
       title="Random walk: with only 10 episodes, roughly tracks the shape of the true values")
ax.set_xticks(range(7)); ax.legend()
fig.savefig(IMG + "/ch07_1_nstep_V10ep.svg", bbox_inches="tight")
plt.show()

n=3, 10에피소드 학습: [ 0.    -0.477 -0.329  0.064  0.236  0.368  0.   ]
참값                 : [ 0.    -0.564 -0.254  0.     0.254  0.564  0.   ]


## 3. 검증: $n=1$이 정확히 TD(0)인가?

$n=1$이면 목표값은 $R_t + \gamma V(s_{t+1})$로, Chapter 6의 TD(0) 갱신과 동일해야 한다. 같은 seed로 "직접 짠 TD(0)"와 `n_step_td(n=1)`를 실행하면 V가 정확히 일치한다.

In [3]:
def td0_reference(n_episodes, alpha, gamma, seed=0):
    """Chapter 6 스타일 TD(0): 매 스텝 즉시 갱신."""
    random.seed(seed)
    V = [0.0] * 7
    for _ in range(n_episodes):
        s = 3
        while True:
            ns, r, done = env_step(s, random.randrange(2))
            G = r + (gamma * V[ns] if not done else 0.0)
            V[s] += alpha * (G - V[s])
            if done:
                break
            s = ns
    return np.array(V)

v_n1 = n_step_td(1, 50, 0.1, 0.9, 7, 3, seed=7)
v_td0 = td0_reference(50, 0.1, 0.9, seed=7)
print("n=1  :", np.round(v_n1, 4))
print("TD(0):", np.round(v_td0, 4))
print("정확히 일치(allclose):", np.allclose(v_n1, v_td0))

n=1  : [ 0.     -0.5458 -0.1611  0.0878  0.3751  0.7817  0.    ]
TD(0): [ 0.     -0.5458 -0.1611  0.0878  0.3751  0.7817  0.    ]
정확히 일치(allclose): True


## 4. U자 곡선: 편향-분산 트레이드오프 (주 실험)

$n \in \{1, 3, 5, 10, 50\}$으로 각각 30개 seed 평균해서, 추정 $V$와 참값의 **RMS 오차**(비터미널 상태 1~5 기준)를 구한다.

- $n=1$(TD(0)): 갱신이 잦지만 추정치 비중이 큼 → 에피소드가 적으면 **편향**이 남음
- $n=50$(사실상 MC, 이 환경의 에피소드 길이는 보통 50보다 짧음): 편향 0이지만 **분산**이 큼
- 중간 어딘가: 오차가 가장 작아 **U자 곡선**

에피소드 예산 10개와 500개 두 가지로 비교한다. (seed가 고정되어 있어 책 본문과 숫자가 약간 다를 수 있지만, U자 모양과 최저점 이동은 같다.)

In [4]:
def rms_over_seeds(n, n_episodes, seeds=30):
    errs = []
    for sd in range(seeds):
        V = n_step_td(n, n_episodes, 0.1, 0.9, 7, 3, seed=sd)
        errs.append(np.sqrt(np.mean((V[1:6] - V_TRUE[1:6]) ** 2)))
    return float(np.mean(errs))

n_list = [1, 3, 5, 10, 50]
res = {ep: {n: rms_over_seeds(n, ep) for n in n_list} for ep in (10, 500)}
for ep in (10, 500):
    best = min(n_list, key=lambda n: res[ep][n])
    print(f"episodes={ep}:")
    for n in n_list:
        tag = "  <- 최저점" if n == best else ""
        print(f"  n={n:3d}: RMS {res[ep][n]:.4f}{tag}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ep, ax in zip((10, 500), axes):
    vals = [res[ep][n] for n in n_list]
    ax.plot(n_list, vals, "o-")
    for n, v in zip(n_list, vals):
        ax.annotate(f"{v:.3f}", (n, v), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=9)
    best = min(n_list, key=lambda n: res[ep][n])
    ax.set_title(f"{ep} episodes (minimum at n={best})")
    ax.set_xlabel("n (number of actual rewards in the target)")
    ax.set_xticks(n_list)
axes[0].set_ylabel("RMS error vs true values (mean of 30 seeds)")
fig.suptitle("Bias-variance trade-off of n-step TD: U-shaped curve (random walk, alpha=0.1, gamma=0.9)", y=1.03)
fig.tight_layout()
fig.savefig(IMG + "/ch07_1_nstep_ucurve.svg", bbox_inches="tight")
plt.show()

episodes=10:
  n=  1: RMS 0.1797
  n=  3: RMS 0.1404  <- 최저점
  n=  5: RMS 0.1641
  n= 10: RMS 0.1831
  n= 50: RMS 0.1930
episodes=500:
  n=  1: RMS 0.1084  <- 최저점
  n=  3: RMS 0.1278
  n=  5: RMS 0.1487
  n= 10: RMS 0.1747
  n= 50: RMS 0.1833


## 5. 결론

- 에피소드 예산이 작으면(10개) 최저점이 중간($n=3$)에 있고, 예산을 늘리면(500개) 최저점이 작은 $n$($n=1$) 쪽으로 **이동** — 데이터가 쌓이면 "갱신이 잦고 분산이 작은" TD(0)의 이점이 살아남.
- 큰 $n$ 쪽 패널티는 예산이 커져도 크게 줄지 않음.
- **"보편적인 최적 n"은 없다** — 검증 데이터로 튜닝해야 하는 하이퍼파라미터이며, 7.2절의 적격흔적은 "n을 고르는 문제" 자체를 피하는 대안이다.